# Tutorial 07: Learning pulsar parameters with a CNN

In the following, we describe how a CNN can be used to estimate the characteristic parameters that describe the birth properties of a population of neutron stars. Here, we focus on obtaining point estimates for the means of the initial spin period and magnetic field distributions in $log_{10}$, i.e., `P_initial_log10_mean` and `B_initial_log10_mean`.

We use a supervised learning approach where the training dataset is composed of simulated neutron star populations represented as $P-\dot{P}$ diagram density maps. These are suitably labeled by the ground truth values of the two parameters `P_initial_log10_mean` and `B_initial_log10_mean`. The network is trained to predict the target value of these labels from the associated synthetic population samples. For the following example, we will use a three channel input formed by three $P-\dot{P}$ density maps resulting from three simulated radio surveys.

To create the dataset of mapped simulations for this training experiment, we can either use Tutorials 04 and 05,  `04_simulation_helper_tutorial.ipynb` and `05_generator_tutorial.ipynb`, respectively, or alternatively we can take advantage of the dataset that is stored in `data/example_generator_magrot`.

The `pypopsyn/learning/train_nn.py` script allows us to perform this training experiment by running the following command:
```commandline
python pypopsyn/learning/train_nn.py --configuration config.json
```
Here, the `config.json` file contains all the information required to optimize the neural network, including the training and validation datasets, the network architecture, the optimization procedure and several others (see documentation for more details). We will use the specific example configuration `tutorials/tutorials_notebooks/conig_train_cnn.json`, but we could also customize another configuration file and provide it to the train script.

Note that as we are using a small dataset in this example, we will adopt the dataset specified in `/data/example_generator_magrot/dataset_test.csv` for both validation and testing purposes. In a "real" experiment, the validation and test datasets should be different to guarantee unbiased and consistent results.

In [ ]:
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

from matplotlib.ticker import ScalarFormatter

import utilities.plot_settings

## Running the training script

To execute the training experiment, we run the shell command from inside this notebook.

In [ ]:
!python ../../pypopsyn/learning/train_nn.py --configuration config_train_cnn.json

## Extracting training results

In the following, we extract the training and validation losses for the two parameters we are predicting to see how they evolve as a function of training epoch during the network optimization process.

NOTE: Make sure to change the time stamps in the variables `train_log_output_path` and `model_output_path` below to where your training results have been saved. Otherwise, the following examples will not work.

Setting the paths to the training logs and saved model.

In [ ]:
with open(f"config_train_cnn.json", "r") as read_file:
    config_train = json.load(read_file)

train_output_path = config_train["trainer"]["save_dir"]
train_log_output_path = (
    f"{train_output_path}/logs/Convolution/20240926_153423/"
)
model_output_path = f"{train_output_path}/models/Convolution/20240926_153423/"

Loading and extracting the training and validation losses.

In [ ]:
with open(f"{train_log_output_path}train_eval_result.json", "r") as read_file:
    train_data = json.load(read_file)

with open(f"{train_log_output_path}validation_result.json", "r") as read_file:
    valid_data = json.load(read_file)

Extracting the number of epochs trained.

In [ ]:
len(train_data.keys())

In [ ]:
epochs = train_data.keys()

train_loss_P_initial_log10_mean = []
valid_loss_P_initial_log10_mean = []
train_loss_B_initial_log10_mean = []
valid_loss_B_initial_log10_mean = []
train_loss_tot = []
valid_loss_tot = []
train_accuracy = []
valid_accuracy = []

for e in epochs:
    train_loss_P_initial_log10_mean.append(
        train_data[e]["P_initial_log10_mean"]
    )
    train_loss_B_initial_log10_mean.append(
        train_data[e]["B_initial_log10_mean"]
    )
    train_loss_tot.append(
        train_data[e]["loss"]
    )
    train_accuracy.append(
        train_data[e]["MetricAccuracyMSE"]
    )
    valid_loss_P_initial_log10_mean.append(
        valid_data[e]["P_initial_log10_mean"]
    )
    valid_loss_B_initial_log10_mean.append(
        valid_data[e]["B_initial_log10_mean"]
    )
    valid_loss_tot.append(
        valid_data[e]["loss"]
    )
    valid_accuracy.append(
        valid_data[e]["MetricAccuracyMSE"]
    )

epochs = np.array(list(epochs), dtype="float64")

Plotting the training and validation losses as a function of training epoch.

In [ ]:
colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]

fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(
    epochs,
    train_loss_tot,
    linestyle="-",
    linewidth=3,
    color=colors[0],
    alpha=0.9,
    rasterized=True,
    label=r"Training",
)
ax.plot(
    epochs,
    valid_loss_tot,
    linestyle="-",
    linewidth=3,
    color=colors[1],
    alpha=0.9,
    rasterized=True,
    label=r"Validation",
)

ax.set_xlabel(r"Training epoch")
ax.set_ylabel(r"MSE loss")
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.legend(frameon=False, loc=0, fontsize=24)

plt.show()

Plotting the training and validation losses as a function of training epoch for the two parameters we set out to predict.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(
    epochs,
    train_loss_P_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color=colors[0],
    alpha=0.9,
    rasterized=True,
    label=r"Training",
)
ax.plot(
    epochs,
    valid_loss_P_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color=colors[1],
    alpha=0.9,
    rasterized=True,
    label=r"Validation",
)

ax.set_xlabel(r"Training epoch")
ax.set_ylabel(r"$\mu_{\log P}$ ($P$ in s) MSE loss")
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.legend(frameon=False, loc=0, fontsize=24)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))

ax.plot(
    epochs,
    train_loss_B_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color=colors[0],
    alpha=0.9,
    rasterized=True,
    label=r"Training",
)
ax.plot(
    epochs,
    valid_loss_B_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color=colors[1],
    alpha=0.9,
    rasterized=True,
    label=r"Validation",
)

ax.set_xlabel(r"Training epoch")
ax.set_ylabel(r"$\mu_{\log B}$ ($B$ in G) MSE loss")
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.legend(frameon=False, loc=0, fontsize=24)

plt.show()

## Performing inference with the trained model

Once a network has been trained, it can be used to infer on an unseen dataset of generated maps and extract the relevant pulsar population parameters. The script `pypopsyn/learning/infer.py` allows us to take an experiment configuration file, a pre-trained model, and a dataset, and run inference on selected samples from that dataset.

Here, we show the performance of the model when predicting the two parameters in the test dataset samples.

We first define the path to the trained model.

In [ ]:
trained_model_path = f"{model_output_path}/best_model_trial1.pth"

We then run the inference script as a shell command from inside this notebook.

In [ ]:
# Construct the command.
command = f"python ../../pypopsyn/learning/infer_nn.py --configuration config_train_cnn.json --trained_model {trained_model_path}"

# Execute the command.
!{command}

## Extracting inference results

Setting the path to the inference results.

NOTE: Make sure to the time stamp in the variable `inference_results_path` below to where your inference results have been saved. Otherwise, the following example will not work.

In [ ]:
infer_output_path = config_train["infer"]["save_dir"]
inference_results_path = (
    f"{infer_output_path}/logs/Convolution/20240905_141137"
)

Loading the inference result from the `.csv` file and converting them to NumPy arrays.

In [ ]:
data = pd.read_csv(f"{inference_results_path}/inference_results.csv")
target_P_initial_log10_mean = data["target:P_initial_log10_mean"].to_numpy()
prediction_P_initial_log10_mean = data[
    "predicted:P_initial_log10_mean"
].to_numpy()
target_B_initial_log10_mean = data["target:B_initial_log10_mean"].to_numpy()
prediction_B_initial_log10_mean = data[
    "predicted:B_initial_log10_mean"
].to_numpy()

Plotting the predictions vs. the ground truths for the two parameters for the four simulation samples in our test dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.set_xlabel(r"Target $\mu_{\log P}$ ($P$ in s)")
ax.set_ylabel(r"Predicted $\mu_{\log P}$ ($P$ in s)")

ax.scatter(
    target_P_initial_log10_mean,
    prediction_P_initial_log10_mean,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=50,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    target_P_initial_log10_mean,
    target_P_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color="tab:red",
    alpha=1.0,
    rasterized=True,
    zorder=1,
)

fig, ax = plt.subplots(figsize=(15, 8))

ax.set_xlabel(r"Target $\mu_{\log B}$ ($B$ in G)")
ax.set_ylabel(r"Predicted $\mu_{\log B}$ ($B$ in G)")

ax.scatter(
    target_B_initial_log10_mean,
    prediction_B_initial_log10_mean,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=50,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    target_B_initial_log10_mean,
    target_B_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color="tab:red",
    alpha=1.0,
    rasterized=True,
    zorder=1,
)
plt.show()